# STAT 764 · Meeting 21 — Thirty-nine measures, a few directions

**Tuesday, November 17**

A new unit: **structure without an answer key**. Until now every model had an outcome to check against: did the
patient come back? Today's tool, **principal component analysis (PCA)**, has none. It takes many measures that move
together and finds the few directions they mostly move in.

The data: **County Health Rankings 2025**, 39 measures for each of about 3,000 US counties (health, behavior, care,
money, housing, environment and who lives there). Three questions run through today:

| the question | today |
|---|---|
| Why do 39 measures collapse into a few directions? | Section 1 · two measures, one direction |
| What are the directions, and what do they mean? | Section 2 · reading the loadings |
| Does compressing help a prediction? | Section 3 · PCA inside a pipeline |

The page `pca.html` in this folder has the same ideas with pictures. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import load

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# County Health Rankings 2025: one row per county. The file's second row holds variable codes, not a county, so skip
# it; the rows with County FIPS Code 0 are whole states and the nation, so drop them.
raw = load("county_health_2025.csv", low_memory=False, skiprows=[1])
raw = raw[pd.to_numeric(raw["County FIPS Code"], errors="coerce") != 0]

# 39 county RATES (shares, averages, per-person figures). No counts such as population, and no death measures, which
# would nearly restate the outcome we predict in Section 3, life expectancy.
MEASURES = [
    "Poor or Fair Health", "Poor Physical Health Days", "Poor Mental Health Days", "Frequent Physical Distress",
    "Frequent Mental Distress", "Diabetes Prevalence", "Adult Obesity", "Adult Smoking", "Physical Inactivity",
    "Excessive Drinking", "Insufficient Sleep", "Food Insecurity", "% Disability: Functional Limitations",
    "Uninsured", "Uninsured Adults", "Uninsured Children", "Flu Vaccinations", "Mammography Screening",
    "Preventable Hospital Stays", "High School Completion", "Some College", "Unemployment", "Children in Poverty",
    "Median Household Income", "Income Inequality", "Children in Single-Parent Households", "Social Associations",
    "Broadband Access", "Homeownership", "Severe Housing Problems", "Severe Housing Cost Burden",
    "Air Pollution: Particulate Matter", "Driving Alone to Work", "Long Commute - Driving Alone", "% Rural",
    "% 65 and Older", "% Below 18 Years of Age", "% Non-Hispanic White", "% Hispanic"]
OUTCOME = "Life Expectancy"

counties = raw[["State Abbreviation", "Name"]].rename(columns={"State Abbreviation": "state", "Name": "county"})
for m in MEASURES + [OUTCOME]:
    counties[m] = pd.to_numeric(raw[f"{m} raw value"], errors="coerce")    # each measure's "raw value" column

# Today uses the counties with every measure (Thursday is about the ones that are missing some).
complete = counties.dropna(subset=MEASURES + [OUTCOME]).reset_index(drop=True)
X, y, state = complete[MEASURES], complete[OUTCOME].to_numpy(), complete["state"].to_numpy()
print(f"{len(counties):,} counties · {len(complete):,} with all {len(MEASURES)} measures and life expectancy")

## 1. Two measures, one direction

Start small. Counties with more adult smokers also tend to have more physically inactive adults. Put both on the
same scale (**standardize**: subtract the average, divide by the standard deviation), and the cloud of counties
stretches along one line. That line is the **first principal component**: the direction in which the counties
differ most. How much of the spread lies along it is its **share of the variance**.

In [ ]:
two = StandardScaler().fit_transform(complete[["Adult Smoking", "Physical Inactivity"]])  # both on the same scale
pca_two = PCA().fit(two)
print(f"correlation: {np.corrcoef(two.T)[0, 1]:.2f}")
print(f"share of the spread along the first direction: {pca_two.explained_variance_ratio_[0]:.1%}")
print(f"the first direction, in (smoking, inactivity): {pca_two.components_[0].round(3)}")

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.scatter(two[:, 0], two[:, 1], s=4, alpha=0.3, color="0.4")
d = pca_two.components_[0] * 3                                 # draw the first direction as an arrow from the center
ax.annotate("", xy=d, xytext=-d, arrowprops=dict(arrowstyle="->", lw=2.5, color="black"))
ax.set(xlabel="adult smoking (standardized)", ylabel="physical inactivity (standardized)", aspect="equal")
plt.show()

The two measures agree to a correlation of 0.79, and 90% of the spread lies along one direction that weighs them
equally. One number per county, its position along that line, keeps 90% of what the two measures say.

## 2. Thirty-nine measures: what are the directions?

PCA does the same with all 39 standardized measures. It finds the direction of greatest spread, then the next one
at right angles to it, and so on. Each direction is a **component**; its **loadings** are the weights it puts on
the 39 measures, and a county's **score** is where it sits along the component.

In [ ]:
scale = StandardScaler().fit(X)
pca = PCA().fit(scale.transform(X))                 # all 39 components
share = pca.explained_variance_ratio_
print("share of variance, components 1-6:", " · ".join(f"{s:.1%}" for s in share[:6]))
print(f"components needed for 80% of the variance: {np.searchsorted(np.cumsum(share), 0.80) + 1}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, 16), share[:15], color="0.5")
ax.set(xlabel="component", ylabel="share of the variance", xticks=range(1, 16))
plt.show()

# The loadings of component 1, largest first (by size: a negative weight matters as much as a positive one).
loadings = pd.DataFrame(pca.components_[:3].T, index=MEASURES, columns=["PC1", "PC2", "PC3"])
print("\ncomponent 1's largest loadings:")
print(loadings["PC1"].sort_values(key=abs, ascending=False).head(8).round(3).to_string())

Component 1 carries 39% of the variance, and every one of its largest loadings is about sickness and poverty: poor
or fair health, disability, physical distress, diabetes, inactivity, children in poverty. A county far up this
direction is worse off on all of them at once. (Its sign is arbitrary: PCA could as well have pointed it the other
way.)

**Under the hood: Z = XΦ.** As on the page: the loading matrix Φ holds the eigenvectors of the standardized
measures' covariance matrix S, each component's variance is its eigenvalue, and the scores are Z = XΦ. The next
cell does it by hand and checks it against `pca`.

In [ ]:
# PCA by hand: the eigen-decomposition of the 39 x 39 covariance matrix, then Z = X Phi.
standardized = scale.transform(X)                    # X, standardized
S = np.cov(standardized, rowvar=False)               # covariance matrix (with standardized columns: the correlations)
eigenvalues, Phi = np.linalg.eigh(S)                 # eigh is for symmetric matrices; it returns the smallest first
order = np.argsort(eigenvalues)[::-1]                # largest first, the order PCA uses
eigenvalues, Phi = eigenvalues[order], Phi[:, order] # Phi: one column per component, one row per measure
Z = standardized @ Phi                               # Z = X Phi: every county's scores on all 39 components

print("eigenvalues 1-3:     ", eigenvalues[:3].round(2))
print("PCA's variances 1-3: ", pca.explained_variance_[:3].round(2))
# The same direction gives a dot product of +1 or -1 (unit vectors; the sign is arbitrary).
print(f"column 1 of Phi · component 1's loadings: {Phi[:, 0] @ pca.components_[0]:+.4f}")
print("Z = X Phi matches pca.transform, up to each column's sign:",
      np.allclose(np.abs(Z), np.abs(pca.transform(standardized))))
print(f"total variance (sum of the eigenvalues): {eigenvalues.sum():.1f} · component 1's share: "
      f"{eigenvalues[0] / eigenvalues.sum():.1%}")

In [ ]:
scores = pca.transform(scale.transform(X))
complete["PC1"] = scores[:, 0]
print("highest on component 1:")
print(complete.nlargest(3, "PC1")[["county", "state", "PC1", OUTCOME]].round(1).to_string(index=False))
print("\nlowest on component 1:")
print(complete.nsmallest(3, "PC1")[["county", "state", "PC1", OUTCOME]].round(1).to_string(index=False))
print(f"\ncorrelation of component 1 with life expectancy: {np.corrcoef(scores[:, 0], y)[0, 1]:.2f}")

## 3. Does compressing help a prediction?

A tempting use: replace the 39 measures with a few components and predict from those. Do it the way the course
always does, with every step **inside the pipeline**: standardize, PCA, then linear regression, all learned on the
training counties only.

One more choice: split **by state**. Counties in one state share laws, Medicaid rules and climate, so a random split
would put near-copies of the test counties into training (the grouped split from Meeting 4, where whole patients were
held out). `GroupKFold` keeps
each state's counties together. The score is the RMSE: how many years of life expectancy the prediction misses by,
on counties from states it did not train on.

In [ ]:
cv = GroupKFold(n_splits=5)                        # 5 folds, each holding out whole states


def cv_rmse(model, data=X):
    """Average RMSE, in years, over the 5 state-grouped folds."""
    return -cross_val_score(model, data, y, groups=state, cv=cv, scoring="neg_root_mean_squared_error").mean()


five = make_pipeline(StandardScaler(), PCA(n_components=5), LinearRegression())
print(f"5 components:      {cv_rmse(five):.2f} years")
print(f"predict the mean:  {np.sqrt(np.mean((y - y.mean()) ** 2)):.2f} years (life expectancy varies by this much)")

Five components cut the miss from 3.5 years to 2.0. Whether that beats the 39 measures themselves is the studio's
third question.

## Studio

**Everything you need is in this notebook.** Sections 1–3 show each idea once; the studio asks what the
directions mean, what happens without standardizing, and whether compressing pays.

| | what you practice | about |
|---|---|---|
| **Question 1** | naming components the way the page's reveal "Interpreting the components" (end of Part 2) does | 14 minutes |
| **Question 2** | why the scale of each measure decides what PCA finds | 8 minutes |
| **Question 3** | whether a few components predict as well as all the measures | 10 minutes |

**All three questions count.** That is about 32 minutes in class; finish the rest at home: the notebook is due 24
hours after class.

**How to work.**

1. Run the notebook from the top (**Run All**) first: it takes a few seconds. The questions use `X`, `y`, `state`,
   `complete`, `MEASURES`, `scale`, `pca`, `loadings`, `scores` and `cv_rmse`.
2. Before each question, **write your guess** as a comment in its cell, and keep it when the answer disagrees.
   The Compare uses it.
3. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
4. Save and upload to Canvas under "Studio notebook — Day 21" by **Wednesday, November 18, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `loadings`, `pca` or `cv_rmse` | Run the notebook from the top (Run All). |
| The largest loadings are all positive, and negative ones are missing | Sort by size: `.sort_values(key=abs, ascending=False)`. |
| You get component 3 when you wanted component 2 | Python counts from 0: component 2 is `scores[:, 1]` and `pca.explained_variance_[1]`. |
| `KeyError: 'PC2'` on `complete` | Make the column first: `complete["PC2"] = scores[:, 1]`. |
| `ValueError: could not convert string to float` | You passed `complete` with the county names; use `X` (the 39 measures only). |
| Component 1's share is 99% or more | That is question 2's point: PCA without `StandardScaler`. |
| A share prints as 100.0%, or a loading as 1.0 | It is rounded: print more digits, `:.2%` or `.round(4)`. |
| The RMSE is negative | `cross_val_score` returns minus the RMSE; `cv_rmse` already flips the sign. |

### 1. Name the components

- **(a)** Components 2 and 3 as correlations with the measures (a loading times √λₘ, as in the page's reveal "Interpreting the
  components"). For
  component 2: `(loadings["PC2"] * np.sqrt(pca.explained_variance_[1])).sort_values(key=abs, ascending=False)`,
  the 8 largest. For component 3, use `"PC3"` and `pca.explained_variance_[2]`.
- **(b)** The three counties at each end of each: `complete["PC2"] = scores[:, 1]`, then
  `complete.nlargest(3, "PC2")[["county", "state"]]` and `complete.nsmallest(3, "PC2")[["county", "state"]]`. The
  same for `"PC3"` with `scores[:, 2]`.
- **(c)** How every component relates to life expectancy, as Section 2 printed for component 1:
  `r = pd.Series([np.corrcoef(scores[:, m], y)[0, 1] for m in range(39)], index=range(1, 40))`, then the 5 largest
  by size, `r.sort_values(key=abs, ascending=False).head(5)`.

Guess first: will the second-biggest direction also be about health?

**Answer in three or four sentences:** a name for components 2 and 3, each with both ends (what high scores mean,
what low scores mean), and whether the counties at the ends fit your names. What do component 3's three largest
measures have in common, and what does that say about look-alike columns (Meeting 20, Part 1: columns that are
near-copies of each other)? Which components are most related to life expectancy: is that the order of their shares
of variance?

### 2. Without standardizing

- **(a)** Fit `PCA()` to `X` itself, with no `StandardScaler`: `raw = PCA().fit(X)`. Print component 1's share of the
  variance and its 3 largest loadings, by size: `pd.Series(raw.components_[0], index=MEASURES)`.
- **(b)** Measure income in **thousands** of dollars instead: `X_k = X.assign(**{"Median Household Income":
  X["Median Household Income"] / 1000})`. Fit `PCA()` to `X_k` and print the same two things.
- **(c)** Print the standard deviation of each measure in `X` (`X.std()`), largest first, top 5.

Guess first: without standardizing, which measure will component 1 be about?

**Answer in two or three sentences:** what does PCA find without standardizing, and why does changing income's
units change the answer?

### 3. Does compressing pay?

- **(a)** `cv_rmse` for linear regression on all 39 measures: `make_pipeline(StandardScaler(), LinearRegression())`.
- **(b)** `cv_rmse` with k components, for every k from 1 to 39 (about 10 seconds):
  `rmse = {k: cv_rmse(make_pipeline(StandardScaler(), PCA(n_components=k), LinearRegression())) for k in range(1, 40)}`.
  Print one table: k = 1, 2, 5, 10 and 20 beside their RMSE, with the 39 measures as the last row.
- **(c)** The fewest components that come within 0.1 years of the 39 measures. And which component, when added,
  lowers the RMSE the most after component 1: `pd.Series(rmse).diff().nsmallest(3)`.

Guess first: how many components will predict as well as all 39 measures?

**Answer in two or three sentences:** does compressing help this prediction? The shares of variance were computed
without life expectancy: is their order the order in which components help predict it? Use your answer to 1(c).

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of next class

We compare answers at the start of next class (Thursday, November 19), built from the notebooks you upload. So
leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: your two names (both ends), and the components most related to life expectancy.
3. Question 3: your table, the number of components within 0.1 years, and the component that helped most after the
   first.

## Exit ticket

> A county health department hands you a report built on "the first principal component of 39 health measures."
> In two or three sentences: what two questions would you ask about it before believing what it says?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook — Day 21". Your
two studio points need this answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ISLP §12.2** — this week's reading: principal components, the scree plot, and why scaling matters.

📗 **3Blue1Brown, *Essence of Linear Algebra*, chapters 13–14** (free online) — change of basis and eigenvectors:
the geometry under PCA, in pictures.

📗 **ESL §14.5** — principal components, curves and surfaces; the more mathematical treatment.

🎓 **Factor analysis** — a model that treats the measures as noisy readings of a few hidden factors. It looks like
PCA and answers a different question; it needs more room than this course has.

🎓 **Sparse PCA** — components that load on only a few measures, so they are easier to name. Worth reading once
you have named a few components by hand.